In [3]:
# !pip install sentence-transformers
# !pip install faiss-cpu
# !pip install chromadb

In [4]:
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

import faiss

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [5]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("SBERT Model Loaded")

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


SBERT Model Loaded


In [6]:
documents = [
    "Learn Python programming from scratch.",
    "Artificial Intelligence for beginners.",
    "Machine Learning fundamentals.",
    "Deep Learning with PyTorch.",
    "Natural Language Processing basics.",
    "Best pizza recipes at home.",
    "Top tourist destinations in Europe.",
    "Introduction to Data Science.",
]

for idx, doc in enumerate(documents):
    print(f"{idx}: {doc}")

0: Learn Python programming from scratch.
1: Artificial Intelligence for beginners.
2: Machine Learning fundamentals.
3: Deep Learning with PyTorch.
4: Natural Language Processing basics.
5: Best pizza recipes at home.
6: Top tourist destinations in Europe.
7: Introduction to Data Science.


In [7]:
document_embeddings = model.encode(documents)

print(document_embeddings.shape)

(8, 384)


In [8]:
document_embeddings = np.array(document_embeddings).astype("float32")

print(document_embeddings.dtype)

float32


In [9]:
dimension = document_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

print(index)

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x00000248199AF9C0> >


In [10]:
index.add(document_embeddings)

print("Vectors Stored:", index.ntotal)

Vectors Stored: 8


In [11]:
query = "How can I start learning AI?"

query_embedding = model.encode([query]).astype("float32")

print(query_embedding.shape)

(1, 384)


In [12]:
k = 3

distances, indices = index.search(query_embedding, k)

print("Distances:")
print(distances)

print("\nIndices:")
print(indices)

Distances:
[[0.69669163 1.1381791  1.3997679 ]]

Indices:
[[1 0 2]]


In [13]:
def faiss_search(query, model, index, documents, top_k=5):

    query_embedding = model.encode([query]).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for distance, idx in zip(distances[0], indices[0]):

        results.append({"Document": documents[idx], "Distance": distance})

    return pd.DataFrame(results)

In [14]:
faiss_search(
    query="Learn Deep Learning", model=model, index=index, documents=documents, top_k=3
)

,Document,Distance
0,Deep Learning with PyTorch.,0.626725
1,Artificial Intelligence for beginners.,0.975590
2,Machine Learning fundamentals.,1.056352


In [15]:
import chromadb

In [16]:
client = chromadb.Client()

collection = client.create_collection(name="sentence_similarity_demo")

print("Collection Created")

Collection Created


In [17]:
collection.add(
    documents=documents,
    embeddings=document_embeddings.tolist(),
    ids=[str(i) for i in range(len(documents))],
)

In [18]:
query_embedding = model.encode(query).tolist()

results = collection.query(query_embeddings=[query_embedding], n_results=3)

results

{'ids': [['1', '0', '2']],
 'embeddings': None,
 'documents': [['Artificial Intelligence for beginners.',
   'Learn Python programming from scratch.',
   'Machine Learning fundamentals.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None, None]],
 'distances': [[0.696691632270813, 1.138179063796997, 1.3997679948806763]]}